# Обучение CatBoost Reranker

Идея: ALS-кандидаты (top-100) — это первая стадия (retrieval).  
Reranker переранжирует их, используя дополнительные признаки: мету пользователей, мету айтемов и ALS-скор.

1. Загружаем данные и обученную ALS-модель из Redis
2. Генерируем кандидатов (top-100 ALS) для каждого пользователя
3. Строим признаки и таргет
4. Обучаем CatBoost
5. Логируем модель в MLflow

In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, log_loss
import mlflow
import redis

## 1. Загрузка данных

In [ ]:
interactions_raw = pd.read_csv("../../../data/interactions.csv")
items_df = pd.read_csv("../../../data/items.csv")
users_df = pd.read_csv("../../../data/users.csv")

# Те же фильтрации и сплит, что и в train_als
interactions = interactions_raw.sample(n=100000, random_state=41)
interactions = interactions[interactions["watched_pct"] >= 10].copy()
interactions["last_watch_dt"] = pd.to_datetime(interactions["last_watch_dt"])
interactions = interactions.sort_values("last_watch_dt")

test_df = interactions.groupby("user_id").tail(1)
train_df = interactions.drop(test_df.index)

# Множество тестовых пар
test_pairs = set(zip(test_df["user_id"], test_df["item_id"]))

print(f"Train: {len(train_df)}, Test: {len(test_df)}")
print(f"Test pairs: {len(test_pairs)}")

## 2. Загружаем ALS-эмбеддинги из Redis

In [ ]:
r = redis.Redis(host="localhost", port=6379, password="recsys_redis_pass", decode_responses=True)

item_ids = json.loads(r.get("als:item_ids"))

# Загружаем item эмбеддинги
pipe = r.pipeline()
for iid in item_ids:
    pipe.get(f"als:item:{iid}")
raw = pipe.execute()

item_emb_dict = {}
for iid, emb_raw in zip(item_ids, raw):
    if emb_raw:
        item_emb_dict[iid] = np.array(json.loads(emb_raw), dtype=np.float32)

item_emb_matrix = np.stack([item_emb_dict[iid] for iid in item_ids])
item_id_to_pos = {iid: pos for pos, iid in enumerate(item_ids)}

print(f"Item embeddings: {item_emb_matrix.shape}")

## 3. Подготовка фичей

Для каждого пользователя берём top-100 ALS-кандидатов.  
Для обучения берём сэмпл пользователей (иначе датасет будет огромным).

In [ ]:
# Подготовка мета-фичей
# Users
users_df = users_df.set_index("user_id")

# Items — числовые и категориальные фичи
items_df["n_genres"] = items_df["genres"].fillna("").apply(lambda x: len(x.split(", ")) if x else 0)
items_df["n_countries"] = items_df["countries"].fillna("").apply(lambda x: len(x.split(", ")) if x else 0)
items_df["n_actors"] = items_df["actors"].fillna("").apply(lambda x: len(x.split(", ")) if x else 0)
items_feat = items_df.set_index("item_id")[
    ["content_type", "release_year", "age_rating", "for_kids", "n_genres", "n_countries", "n_actors"]
].copy()
items_feat["release_year"] = items_feat["release_year"].fillna(0).astype(int)
items_feat["age_rating"] = items_feat["age_rating"].fillna(0).astype(int)
items_feat["content_type"] = items_feat["content_type"].fillna("unknown")
items_feat["for_kids"] = items_feat["for_kids"].fillna("unknown")

print(f"Users features: {users_df.shape}")
print(f"Items features: {items_feat.shape}")

In [ ]:
N_CANDIDATES = 100
SAMPLE_USERS = 5000  # сэмпл пользователей для обучения

# Сэмплируем пользователей из теста
sample_user_ids = test_df["user_id"].drop_duplicates().sample(SAMPLE_USERS, random_state=42).values

rows = []
for uid in sample_user_ids:
    # Эмбеддинг пользователя
    u_emb_raw = r.get(f"als:user:{uid}")
    if not u_emb_raw:
        continue
    u_emb = np.array(json.loads(u_emb_raw), dtype=np.float32)

    # ALS-скоры по всем айтемам, top-N кандидатов
    scores = item_emb_matrix @ u_emb
    top_indices = np.argsort(scores)[::-1][:N_CANDIDATES]

    for pos, idx in enumerate(top_indices):
        iid = item_ids[idx]
        als_score = float(scores[idx])
        label = 1 if (uid, iid) in test_pairs else 0
        rows.append({"user_id": uid, "item_id": iid, "als_score": als_score, "als_rank": pos, "label": label})

candidates_df = pd.DataFrame(rows)
print(f"Candidates: {candidates_df.shape}")
print(f"Positive rate: {candidates_df['label'].mean():.4f}")

In [ ]:
# Джойним мета-фичи
candidates_df = candidates_df.merge(users_df, left_on="user_id", right_index=True, how="left")
candidates_df = candidates_df.merge(items_feat, left_on="item_id", right_index=True, how="left")

# Заполняем NaN в категориальных фичах — CatBoost не принимает NaN для cat_features
for col in ["age", "income", "sex", "content_type", "for_kids"]:
    candidates_df[col] = candidates_df[col].fillna("unknown").astype(str)

# Заполняем NaN в числовых фичах
for col in ["kids_flg", "release_year", "age_rating", "n_genres", "n_countries", "n_actors"]:
    candidates_df[col] = candidates_df[col].fillna(0)

print(f"NaN check:\n{candidates_df.isna().sum()[candidates_df.isna().sum() > 0].to_string() or 'No NaN'}")
candidates_df.head()

In [ ]:
FEATURE_COLS = [
    # ALS
    # "als_score", "als_rank",
    # User
    "age", "income", "sex", "kids_flg",
    # Item
    "content_type", "release_year", "age_rating", "for_kids",
    "n_genres", "n_countries", "n_actors",
]

CAT_FEATURES = ["age", "income", "sex", "content_type", "for_kids"]

TARGET = "label"

# Train/val split: 80/20 по пользователям
unique_sample_users = candidates_df["user_id"].unique()
np.random.seed(42)
np.random.shuffle(unique_sample_users)
split = int(0.8 * len(unique_sample_users))
train_users = set(unique_sample_users[:split])

train_mask = candidates_df["user_id"].isin(train_users)
df_train = candidates_df[train_mask]
df_val = candidates_df[~train_mask]

print(f"Train: {len(df_train)} ({df_train['label'].mean():.4f} pos rate)")
print(f"Val:   {len(df_val)} ({df_val['label'].mean():.4f} pos rate)")

## 4. Обучение CatBoost

In [ ]:
train_pool = Pool(
    df_train[FEATURE_COLS],
    label=df_train[TARGET],
    cat_features=CAT_FEATURES,
)
val_pool = Pool(
    df_val[FEATURE_COLS],
    label=df_val[TARGET],
    cat_features=CAT_FEATURES,
)

model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
)

model.fit(train_pool, eval_set=val_pool)

In [ ]:
# Метрики на валидации
val_preds = model.predict_proba(val_pool)[:, 1]
val_auc = roc_auc_score(df_val[TARGET], val_preds)
val_logloss = log_loss(df_val[TARGET], val_preds)

print(f"Val AUC:     {val_auc:.4f}")
print(f"Val LogLoss: {val_logloss:.4f}")

In [ ]:
# Feature importance
fi = pd.Series(model.get_feature_importance(), index=FEATURE_COLS).sort_values(ascending=False)
print("Feature importance:")
print(fi.to_string())

## 5. Логирование в MLflow

Сохраняем модель через `mlflow.catboost.log_model` — это позволит загрузить её при инференсе через `mlflow.catboost.load_model`.

In [ ]:
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("reranker")

with mlflow.start_run(run_name="catboost_reranker"):
    mlflow.log_params({
        "model_type": "CatBoostClassifier",
        "iterations": 500,
        "depth": 6,
        "learning_rate": 0.05,
        "n_candidates": N_CANDIDATES,
        "sample_users": SAMPLE_USERS,
        "features": ", ".join(FEATURE_COLS),
        "cat_features": ", ".join(CAT_FEATURES),
    })

    mlflow.log_metrics({
        "val_auc": val_auc,
        "val_logloss": val_logloss,
    })

    # Сохраняем модель как mlflow-артефакт (нативный формат CatBoost)
    mlflow.catboost.log_model(
        cb_model=model,
        artifact_path="reranker_model",
    )

    # Сохраняем список фичей для инференса
    mlflow.log_dict(
        {"feature_cols": FEATURE_COLS, "cat_features": CAT_FEATURES},
        artifact_file="feature_config.json",
    )

    run_id = mlflow.active_run().info.run_id
    print(f"Run ID: {run_id}")
    print(f"Model URI: runs:/{run_id}/reranker_model")

In [ ]:
# Сохраняем run_id в Redis, чтобы FastAPI мог найти модель
r.set("reranker:mlflow_run_id", run_id)
print(f"Saved run_id={run_id} to Redis key 'reranker:mlflow_run_id'")

In [ ]:
# Проверяем: загружаем модель из MLflow и делаем предикт
loaded_model = mlflow.catboost.load_model(f"runs:/{run_id}/reranker_model")

test_preds = loaded_model.predict_proba(val_pool)[:, 1]
print(f"Loaded model AUC: {roc_auc_score(df_val[TARGET], test_preds):.4f}")
print("MLflow инференс работает!")

## 6. Сохраняем мета-фичи пользователей и айтемов в Redis

Чтобы FastAPI мог собрать фичи для реранкера в реальном времени.

In [ ]:
# User features -> Redis
# Заполняем NaN перед сериализацией
users_for_redis = users_df.copy()
for col in users_for_redis.columns:
    if users_for_redis[col].dtype == object:
        users_for_redis[col] = users_for_redis[col].fillna("unknown")
    else:
        users_for_redis[col] = users_for_redis[col].fillna(0)

pipe = r.pipeline()
for uid, row in users_for_redis.iterrows():
    pipe.set(f"user_features:{uid}", json.dumps(row.to_dict()))
pipe.execute()
print(f"Saved {len(users_for_redis)} user feature sets")

# Item features -> Redis
# items_feat уже почищен от NaN выше, но на всякий случай
items_for_redis = items_feat.copy()

for col in items_for_redis.columns:
    if items_for_redis[col].dtype == object:
        items_for_redis[col] = items_for_redis[col].fillna("unknown")
    else:
        items_for_redis[col] = items_for_redis[col].fillna(0)

pipe = r.pipeline()
for iid, row in items_for_redis.iterrows():
    d = {k: (int(v) if isinstance(v, (np.integer,)) else v) for k, v in row.to_dict().items()}
    pipe.set(f"item_features:{iid}", json.dumps(d))
pipe.execute()
print(f"Saved {len(items_for_redis)} item feature sets")

In [ ]:
# Проверка
# Берём пользователя, который точно есть в users_df
test_uid = next(uid for uid in sample_user_ids if int(uid) in users_for_redis.index)
test_iid = item_ids[0]

print("User features sample:", json.loads(r.get(f"user_features:{test_uid}")))
print("Item features sample:", json.loads(r.get(f"item_features:{test_iid}")))
print(f"\nReranker run_id: {r.get('reranker:mlflow_run_id')}")
print("\nГотово! Модель в MLflow, фичи в Redis.")

## 7. Экспорт модели для Triton Inference Server

Сохраняем `model.cbm` в model repository Triton.

In [ ]:
triton_model_path = "../../../infra/triton_models/catboost_reranker/1/model.cbm"
model.save_model(triton_model_path)
print(f"Saved model.cbm to {triton_model_path}")

In [ ]:
df_train[FEATURE_COLS]